In [ ]:
!pip install mne kaggle

In [26]:
import os

os.environ['KAGGLE_USERNAME'] = 'yuvikaagarwal2105'
os.environ['KAGGLE_KEY'] = 'KGAT_945517875de4b2e3ee830b5e3ce65707'

In [ ]:
!kaggle datasets download -d brianleung2020/eeg-motor-movementimagery-dataset

In [ ]:
!unzip -o eeg-motor-movementimagery-dataset.zip

In [ ]:
import mne

file = "files/S001/S001R04.edf"
raw = mne.io.read_raw_edf(file, preload=True)

print(raw.info)

In [ ]:
raw.plot()

In [ ]:
raw.filter(7., 30.)

In [ ]:
raw.notch_filter(freqs=50)

In [ ]:
events, event_id = mne.events_from_annotations(raw)

epochs = mne.Epochs(raw, events, event_id=event_id,
                    tmin=0, tmax=4, baseline=None)

In [ ]:
import numpy as np

X = epochs.get_data()

X = (X - np.mean(X)) / np.std(X)

In [ ]:
events, event_id = mne.events_from_annotations(raw)

epochs = mne.Epochs(raw, events, event_id=event_id,
                    tmin=0, tmax=4, baseline=None)

In [ ]:
y = epochs.events[:, -1]

print(y)
print("X shape:", X.shape)
print("y shape:", y.shape)

In [37]:
y = y - np.min(y)

In [ ]:
!pip install torch torchvision

import torch
import torch.nn as nn
import numpy as np

In [39]:
y = epochs.events[:, -1]

y = y - 1

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

In [41]:
class EEGModel(nn.Module):
    def __init__(self):
        super(EEGModel, self).__init__()


        self.conv1 = nn.Conv1d(64, 32, kernel_size=3)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(2)


        encoder_layer = nn.TransformerEncoderLayer(d_model=32, nhead=4)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)


        self.fc = nn.Linear(32, 3)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = x.permute(2, 0, 1)
        x = self.transformer(x)

        x = x.mean(dim=0)
        x = self.fc(x)

        return x

In [ ]:
model = EEGModel()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [44]:
y_train = torch.tensor(y_train, dtype=torch.long)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

In [46]:
print(X_train.shape)

torch.Size([24, 64, 641])


In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

epochs = 20

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [49]:
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

In [50]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs, 1)

    correct = (predicted == y_test).sum().item()
    total = y_test.size(0)

    accuracy = correct / total * 100

print("Accuracy:", accuracy)

Accuracy: 50.0
